In [ ]:
import sys, os
sys.path.append(os.path.abspath("../.."))

In [ ]:
import numpy as np
import pandas as pd
from preprocessing.preprocess import prep, append_results, eval_thresholds
from preprocessing.target import ttp_target, hybrid_target
from metrics.Metrics import merged_metrics
name = "AFKS"
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv")

In [ ]:
class TSMixerBlock(nn.Module):
    def init(self, seq_len, n_features, hidden=128, dropout=0.2):
        super().init()
        self.time_mlp = nn.Sequential(
            nn.LayerNorm(n_features),
            nn.Linear(seq_len, seq_len),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.feat_mlp = nn.Sequential(
            nn.LayerNorm(n_features),
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_features),
        )

    def forward(self, x):
        # x: (B,L,F)
        y = x.transpose(1,2)                 # (B,F,L)
        y = self.time_mlp(y).transpose(1,2)  # (B,L,F)
        x = x + y
        x = x + self.feat_mlp(x)
        return x

class TSMixerCls(nn.Module):
    def init(self, seq_len, n_features, n_blocks=4, hidden=128, n_classes=3):
        super().init()
        self.blocks = nn.Sequential(*[TSMixerBlock(seq_len, n_features, hidden) for _ in range(n_blocks)])
        self.head = nn.Sequential(
            nn.LayerNorm(n_features),
            nn.Linear(n_features, n_classes)
        )

    def forward(self, x):
        x = self.blocks(x)
        h = x[:, -1, :]   # last time
        return self.head(h)

def train_tsmixer_ttp(df, train_size, test_size, step, seq_len=64):
    set_seed(42)
    splitter = prep(
        df=df,
        target_fn=ttp_target, target_name="ttp", target_col="TTP_class",
        horizons=[12,24,48],
        train_size=train_size, test_size=test_size, step=step,
        target_kwargs={"n_classes":3},
        scale_cols=[
            "Open","High","Low","Close",
            "Alligator_Jaw","Alligator_Teeth","Alligator_Lips",
            "AO","AddOn_Anchor_Level","AddOn_Size_Pct"
        ]
    )

    for X_train, X_test, y_train, y_test, scaler in splitter:
        tr_ds = SeqDataset(X_train, y_train, seq_len)
        te_ds = SeqDataset(X_test,  y_test,  seq_len)
        tr_loader = DataLoader(tr_ds, batch_size=256, shuffle=True)
        te_loader = DataLoader(te_ds, batch_size=512, shuffle=False)

        model = TSMixerCls(seq_len=seq_len, n_features=X_train.shape[1], n_blocks=4, hidden=128, n_classes=3)

        y_pred = train_torch_classifier(model, tr_loader, te_loader, epochs=20, lr=1e-3)
        y_test_seq = y_test[seq_len-1:]
        metrics = merged_metrics(y_test_seq, y_pred)

        append_results({
            "task_type":"classification",
            "model_name":"TSMixer",
            "model_family":"deep",
            "model_params":{"n_blocks":4,"hidden":128,"seq_len":seq_len},
            "target_name":"ttp","target_variant":"3class","horizons":"12_24_48",
            **metrics
        })